# Sacred -> Techno backend (Colab Pro)

발표 데모용으로 백엔드(FastAPI + Demucs + MusicGen-Melody)를 실행하고, ngrok 고정 도메인으로 외부 URL을 만듭니다.
Vercel이 이 URL을 바라보도록 한 번만 설정해두면, 그 다음부터는 발표 때마다 이 노트북만 실행하면 됩니다.

## 사전 설정 (한 번만)

1. https://dashboard.ngrok.com 에 가입하고 Auth Token을 확인합니다.
2. https://dashboard.ngrok.com/domains 에서 무료 고정 도메인(static domain)을 하나 발급받습니다 (예: `your-name.ngrok-free.app`).
3. 이 Colab 노트북의 왼쪽 사이드바에서 열쇠 모양 아이콘(Secrets)을 누르고 다음 두 개를 등록합니다. 각각 'Notebook access' 토글을 켜주세요.
   - `NGROK_AUTH_TOKEN`: 1번에서 확인한 토큰
   - `NGROK_DOMAIN`: 2번에서 발급받은 도메인 (`https://` 없이, 예: `your-name.ngrok-free.app`)
4. Vercel 프론트엔드 프로젝트 설정에서 `NEXT_PUBLIC_ML_BACKEND_URL` 값을 `https://<2번 도메인>` 으로 설정하고 재배포합니다 (한 번만).

## 발표 직전에 할 일

1. 상단 메뉴에서 런타임 유형을 GPU로 바꿔주세요 (런타임 -> 런타임 유형 변경 -> GPU).
2. 아래 셀을 위에서부터 순서대로 실행하세요.
3. 마지막 셀이 출력하는 URL이 2번에서 등록한 도메인과 같으면 정상입니다.
4. 이 노트북을 끄지 말고, 발표가 완전히 끝날 때까지 켜둔 상태로 두세요.

In [ ]:
import os

if not os.path.exists("/content/MIR"):
    !git clone https://github.com/rocgwang/MIR.git /content/MIR
else:
    !cd /content/MIR && git pull

%cd /content/MIR/backend

In [ ]:
# Colab에는 CUDA 지원 torch가 이미 설치되어 있으므로 그대로 쓰고, 나머지 패키지만 설치합니다.
!pip install -q -r requirements.txt
!pip install -q pyngrok

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import subprocess
import time
import urllib.request

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/MIR/backend",
)

print("서버와 모델 로딩을 기다리는 중입니다. 처음 실행할 때는 모델 다운로드 때문에 몇 분 걸릴 수 있습니다.")
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/openapi.json", timeout=2)
        break
    except Exception:
        time.sleep(5)
print("서버 준비 완료")

In [ ]:
tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
print("Backend URL:", tunnel.public_url)
print("이 URL이 Vercel의 NEXT_PUBLIC_ML_BACKEND_URL 값과 같은지 확인하세요.")

## 발표 종료 후

이 노트북을 끄거나 런타임을 종료하면 터널과 백엔드 연결이 끊깁니다. 그 이후 사이트에서 변환을 시도하면 '변환 서버에 연결할 수 없습니다' 오류가 표시됩니다.
발표가 끝나면 '런타임 > 런타임 연결 해제 및 삭제'로 종료해서 Colab Pro 사용량을 절약하세요.